# 02 — Sentiment de noticias

**Objetivo.** Convertir titulares de noticias a señal numérica (sentiment score) para uso como feature auxiliar en notebooks de índices.

**⚠️ BAJA PRIORIDAD.** El enunciado advierte que *"todas las fechas han sido desplazadas"*, lo que puede hacer que el alineamiento news↔precio sea incorrecto y convierta el sentiment en ruido. Ejecutar solo si: (a) el EDA de `00_carga_y_EDA` muestra correlación news↔retornos, y (b) hay tiempo disponible. En CPU, la inferencia del transformer puede ser muy lenta — **medir tiempo con 10 filas antes de comprometerse**.

**Decisión tomada.** Notebook separado y opcional. No bloquea ningún otro notebook.

**Qué hace.** Carga train/test_news, detecta idioma, corre FinBERT o modelo multilingüe, exporta scores a CSV.

**Qué NO hace.** No entrena modelos de índices ni genera submission.

**Inputs.** `data/train_news.csv`, `data/test_news.csv`

**Output esperado.** `data/sentiment_train.csv`, `data/sentiment_test.csv` (solo si se ejecuta hasta el final).

## 0. GPU workaround + imports

⚠️ El workaround es obligatorio aunque no usemos TF directamente — HuggingFace puede cargar TF internamente.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import time
import numpy as np
import pandas as pd

from utils import load_hackathon_data, DATA_DIR

## 1. Carga y diagnóstico de news

Ver columnas, sample de titulares y detectar el idioma dominante antes de elegir el modelo.

In [ ]:
data = load_hackathon_data(DATA_DIR)

if 'train_news' not in data:
    raise RuntimeError('train_news.csv no encontrado en data/')

train_news = data['train_news']
test_news  = data['test_news']

print('Columnas train_news:', list(train_news.columns))
print('Shape train:', train_news.shape, '  Shape test:', test_news.shape)
print('\nMuestra de titulares:')
# Mostrar las primeras filas de la columna de texto
# (ajustar el nombre de columna según el CSV real)
print(train_news.head(5))

## 2. Selección de modelo de sentiment

- **Titulares en inglés** → `ProsusAI/finbert` (FinBERT, financiero)
- **Titulares en otro idioma** → `nlptown/bert-base-multilingual-uncased-sentiment` u otro multilingüe

Ajustar `MODEL_NAME` según el idioma detectado en la celda anterior.

In [ ]:
# Ajustar según idioma detectado
MODEL_NAME = 'ProsusAI/finbert'   # alternativa: 'nlptown/bert-base-multilingual-uncased-sentiment'
TEXT_COL   = '?'                  # <-- rellenar con el nombre real de la columna de texto

## 3. BENCHMARK DE TIEMPO — OBLIGATORIO antes de inferencia completa

Medir cuánto tarda el modelo con 10 filas en CPU. Si supera ~2 segundos/fila, el dataset completo puede tardar horas y no compensa.

In [ ]:
# Importar pipeline de HuggingFace solo aquí (por si no está instalado)
from transformers import pipeline

sentiment_pipe = pipeline('text-classification', model=MODEL_NAME, device=-1)  # device=-1 = CPU

sample_texts = train_news[TEXT_COL].dropna().head(10).tolist()

t0 = time.time()
sample_results = sentiment_pipe(sample_texts, truncation=True, max_length=512)
elapsed = time.time() - t0

print(f'10 filas en {elapsed:.1f}s  ({elapsed/10:.2f}s/fila)')
print(f'Estimación para {len(train_news)} filas: {elapsed/10*len(train_news)/60:.0f} minutos')
print('\nResultados de muestra:', sample_results[:3])

# STOP si el tiempo estimado es prohibitivo
if elapsed / 10 > 2.0:
    print('\n[AVISO] > 2s/fila — valorar si el tiempo de cómputo compensa antes de continuar')

## 4. Inferencia de sentiment — train y test

Solo ejecutar si el benchmark anterior fue aceptable.

In [ ]:
def run_sentiment(df, text_col, pipe, batch_size=32):
    texts = df[text_col].fillna('').tolist()
    results = pipe(texts, truncation=True, max_length=512, batch_size=batch_size)
    scores = []
    for r in results:
        # FinBERT: label in {positive, negative, neutral}
        sign = 1 if r['label'] == 'positive' else (-1 if r['label'] == 'negative' else 0)
        scores.append(sign * r['score'])
    return scores

train_news['sentiment'] = run_sentiment(train_news, TEXT_COL, sentiment_pipe)
test_news['sentiment']  = run_sentiment(test_news,  TEXT_COL, sentiment_pipe)

print('Distribución de sentiment (train):')
print(train_news['sentiment'].describe())

## 5. Diagnóstico: ¿hay señal? Correlación sentiment↔retornos

In [ ]:
# Alinear sentiment con retornos de índices por fecha y calcular correlación
# (solo si las fechas se alinean correctamente — recordar el aviso de fechas desplazadas)
idx = data['train_indices']
log_rets = np.log(idx).diff().dropna()

sent_series = train_news['sentiment']
combined = pd.concat([log_rets, sent_series.rename('sentiment')], axis=1).dropna()

print('Correlación sentiment↔retornos:')
print(combined.corr()['sentiment'].drop('sentiment').round(4))

## 6. Guardar scores a CSV

In [ ]:
# Solo guardar si la correlación del paso anterior sugiere señal útil
train_news[['sentiment']].to_csv('data/sentiment_train.csv')
test_news[['sentiment']].to_csv('data/sentiment_test.csv')
print('Guardado: data/sentiment_train.csv  y  data/sentiment_test.csv')